In [2]:
"""
Generate CodeT5 Mean Embeddings and Labels (JSON Version)
--------------------------------------------------------
This script converts dataset.json into NumPy arrays of
mean-pooled CodeT5 embeddings (768-dim) and binary labels.
Model: Salesforce/codet5-770m
Output: <project>_embeddings.npy, <project>_labels.npy
"""
import os
import json
import numpy as np
from tqdm import tqdm
from collections import defaultdict

import torch
from torch import no_grad
from transformers import AutoTokenizer, T5EncoderModel


# ==============================
# Configuration
# ==============================
DATA_PATH = "dataset/dataset.json"
OUTPUT_DIR = "dataset/"
MODEL_NAME = "Salesforce/codet5p-770m"  # CodeT5-base
MAX_LEN = 512
BATCH_SIZE = 8
DEVICE = "cpu"  # Use "cuda" if GPU is available

# ==============================
def load_dataset(path):
    print(f"Loading dataset from {path} ...")
    with open(path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    # Group data by project
    grouped_data = defaultdict(list)
    for item in raw_data:
        grouped_data[item["project"]].append(item)
    print(f"Projects found: {list(grouped_data.keys())}")
    return grouped_data

def get_codet5_model():
    
    print(f"Loading CodeT5 model ({MODEL_NAME}) ...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = T5EncoderModel.from_pretrained(MODEL_NAME)
    model = model.to(DEVICE)
    model.eval()
    return tokenizer, model, no_grad, torch

def embed_code_snippets(snippets, tokenizer, model, no_grad, torch):
    """Generate mean-pooled embeddings for each code snippet."""
    all_embeddings = []
    with no_grad():
        for i in tqdm(range(0, len(snippets), BATCH_SIZE)):
            batch = snippets[i:i + BATCH_SIZE]
            inputs = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_LEN
            ).to(DEVICE)
            outputs = model(**inputs)
            # Mean pooling across token dimension
            # CodeT5 output: last_hidden_state [batch, seq_len, hidden_dim]
            mean_embeddings = outputs.last_hidden_state.mean(dim=1)
            all_embeddings.append(mean_embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

def save_embeddings(project, embeddings, labels):
    emb_path = os.path.join(OUTPUT_DIR, f"{project.lower()}_embeddings.npy")
    lab_path = os.path.join(OUTPUT_DIR, f"{project.lower()}_labels.npy")
    np.save(emb_path, embeddings)
    np.save(lab_path, labels)
    print(f"Saved {project}: {embeddings.shape} → {emb_path}, {lab_path}")

# ==============================
if __name__ == "__main__":
    import torch
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Load JSON dataset
    combined_data = load_dataset(DATA_PATH)

    # Load CodeT5
    tokenizer, model, no_grad, torch = get_codet5_model()

    # Process each project separately
    for project_name, samples in combined_data.items():
        print(f"\nProcessing project: {project_name}")
        codes = [s["code"] for s in samples]
        labels = np.array([s["label"] for s in samples])
        embeddings = embed_code_snippets(codes, tokenizer, model, no_grad, torch)
        save_embeddings(project_name, embeddings, labels)

    print("\nAll mean-pooled CodeT5 embeddings (dim=768) generated successfully.")

Loading dataset from dataset/dataset.json ...
Projects found: ['chrome', 'debian']
Loading CodeT5 model (Salesforce/codet5p-770m) ...


/opt/miniconda3/envs/ml/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Processing project: chrome


100%|████████████████████████████████████████████████████████████████████████████████| 555/555 [33:24<00:00,  3.61s/it]


Saved chrome: (4436, 1024) → dataset/chrome_embeddings.npy, dataset/chrome_labels.npy

Processing project: debian


100%|████████████████████████████████████████████████████████████████████████████| 2288/2288 [2:26:38<00:00,  3.85s/it]

Saved debian: (18298, 1024) → dataset/debian_embeddings.npy, dataset/debian_labels.npy

All mean-pooled CodeT5 embeddings (dim=768) generated successfully.
